In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
     .appName("Working with files") \
     .getOrCreate()

with csv file

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving airline_bookings.csv to airline_bookings.csv


In [ ]:
df=spark.read.csv(
    "airline_bookings.csv",
    header=True,
    inferSchema=True
)
df.show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1001|   Aarav Mehta|Hyderabad|    Delhi|            IndiGo|        6500|     Economy|Confirmed|
|      1002|     Sana Khan|Bangalore|   Mumbai|           Vistara|        8200|     Economy|Confirmed|
|      1003|   John Mathew|  Chennai|    Delhi|         Air India|       12000|    Business|Confirmed|
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1006|  Divya Sharma|    Delhi|Hyderabad|            IndiGo|        5900|     Economy|Cancelled|
|      1007|     Imran Ali|     Pune|Bangalore|         Akasa Air|       

In [ ]:
df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- passenger_name: string (nullable = true)
 |-- from_city: string (nullable = true)
 |-- to_city: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- ticket_price: integer (nullable = true)
 |-- travel_class: string (nullable = true)
 |-- status: string (nullable = true)



In [ ]:
df.select("passenger_name", "from_city", "to_city").show()

+--------------+---------+---------+
|passenger_name|from_city|  to_city|
+--------------+---------+---------+
|   Aarav Mehta|Hyderabad|    Delhi|
|     Sana Khan|Bangalore|   Mumbai|
|   John Mathew|  Chennai|    Delhi|
|  Ayesha Begum|Hyderabad|    Dubai|
|    Vikram Rao|   Mumbai|Singapore|
|  Divya Sharma|    Delhi|Hyderabad|
|     Imran Ali|     Pune|Bangalore|
|    Meera Nair|    Kochi|    Dubai|
|     Rohan Das|  Kolkata|    Delhi|
|   Nisha Reddy|Bangalore|   London|
+--------------+---------+---------+



In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
df.filter(F.col("status") == "Cancelled").show()

+----------+--------------+---------+---------+-------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+-------+------------+------------+---------+
|      1006|  Divya Sharma|    Delhi|Hyderabad| IndiGo|        5900|     Economy|Cancelled|
+----------+--------------+---------+---------+-------+------------+------------+---------+



In [ ]:
df.filter(F.col("status") == "Pending").show()

+----------+--------------+---------+---------+------------------+------------+------------+-------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class| status|
+----------+--------------+---------+---------+------------------+------------+------------+-------+
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|Pending|
|      1009|     Rohan Das|  Kolkata|    Delhi|         Air India|        7400|     Economy|Pending|
+----------+--------------+---------+---------+------------------+------------+------------+-------+



In [ ]:
df.filter(F.col("ticket_price") > 20000).show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1008|    Meera Nair|    Kochi|    Dubai|          Emirates|       26000|     Economy|Confirmed|
|      1010|   Nisha Reddy|Bangalore|   London|   British Airways|       62000|    Business|Confirmed|
+----------+--------------+---------+---------+------------------+------------+------------+---------+



In [ ]:
df.filter(F.col("to_city").isin("Dubai", "Singapore", "London")).show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1008|    Meera Nair|    Kochi|    Dubai|          Emirates|       26000|     Economy|Confirmed|
|      1010|   Nisha Reddy|Bangalore|   London|   British Airways|       62000|    Business|Confirmed|
+----------+--------------+---------+---------+------------------+------------+------------+---------+



In [ ]:
total_bookings = df.count()
print(f"Total bookings: {total_bookings}")

Total bookings: 10


In [ ]:
df.groupBy("airline").count().show()

+------------------+-----+
|           airline|count|
+------------------+-----+
|         Akasa Air|    1|
|         Air India|    2|
|   British Airways|    1|
|            IndiGo|    2|
|           Vistara|    1|
|          Emirates|    2|
|Singapore Airlines|    1|
+------------------+-----+



In [ ]:
df.groupBy("status").count().show()

+---------+-----+
|   status|count|
+---------+-----+
|Cancelled|    1|
|  Pending|    2|
|Confirmed|    7|
+---------+-----+



In [ ]:
total_revenue = df.filter(F.col("status") == "Confirmed").agg(F.sum("ticket_price")).collect()[0][0]
print(f"Total revenue from confirmed bookings: {total_revenue}")

Total revenue from confirmed bookings: 147500


In [ ]:
df.groupBy("airline").agg(F.avg("ticket_price").alias("average_price")).show()

+------------------+-------------+
|           airline|average_price|
+------------------+-------------+
|         Akasa Air|       4800.0|
|         Air India|       9700.0|
|   British Airways|      62000.0|
|            IndiGo|       6200.0|
|           Vistara|       8200.0|
|          Emirates|      27000.0|
|Singapore Airlines|      35000.0|
+------------------+-------------+



In [ ]:
max_price = df.agg(F.max("ticket_price")).collect()[0][0]
print(f"Highest ticket price: {max_price}")

Highest ticket price: 62000


In [ ]:
min_price = df.agg(F.min("ticket_price")).collect()[0][0]
print(f"Lowest ticket price: {min_price}")

Lowest ticket price: 4800


In [ ]:
df.orderBy(F.col("ticket_price").desc()).show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1010|   Nisha Reddy|Bangalore|   London|   British Airways|       62000|    Business|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1008|    Meera Nair|    Kochi|    Dubai|          Emirates|       26000|     Economy|Confirmed|
|      1003|   John Mathew|  Chennai|    Delhi|         Air India|       12000|    Business|Confirmed|
|      1002|     Sana Khan|Bangalore|   Mumbai|           Vistara|        8200|     Economy|Confirmed|
|      1009|     Rohan Das|  Kolkata|    Delhi|         Air India|       

In [ ]:
df = df.withColumn("tax", F.col("ticket_price") * 0.05)

In [ ]:
df = df.withColumn("final_price", F.col("ticket_price") + F.col("tax"))

In [ ]:
df = df.withColumn(
    "price_category",
    F.when(F.col("ticket_price") >= 30000, "Premium")
     .when(F.col("ticket_price") >= 10000, "Standard")
     .otherwise("Budget")
)
df.show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+------+-----------+--------------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|   tax|final_price|price_category|
+----------+--------------+---------+---------+------------------+------------+------------+---------+------+-----------+--------------+
|      1001|   Aarav Mehta|Hyderabad|    Delhi|            IndiGo|        6500|     Economy|Confirmed| 325.0|     6825.0|        Budget|
|      1002|     Sana Khan|Bangalore|   Mumbai|           Vistara|        8200|     Economy|Confirmed| 410.0|     8610.0|        Budget|
|      1003|   John Mathew|  Chennai|    Delhi|         Air India|       12000|    Business|Confirmed| 600.0|    12600.0|      Standard|
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|1400.0|    29400.0|      Standard|
|      1005|    Vikram Rao|   Mumbai|Sing

In [ ]:
df.groupBy("price_category").count().show()

+--------------+-----+
|price_category|count|
+--------------+-----+
|       Premium|    2|
|        Budget|    5|
|      Standard|    3|
+--------------+-----+



In [ ]:
confirmed_df = df.filter(F.col("status") == "Confirmed")
confirmed_df.write.mode("overwrite").parquet("confirmed_flights.parquet")

with json file

In [ ]:
%%writefile hotels.json
[
{
"hotel_id": 201,
"hotel_name": "Pearl Grand",
"city": "Hyderabad",
"category": "Business",
"rating": 4.4,
"rooms_available": 25,
"price_per_night": 4500,
"amenities": ["wifi", "breakfast", "gym"],
"contact": {
"phone": "9876500011",
"email": "pearlgrand@mail.com"
}
},
{
"hotel_id": 202,
"hotel_name": "Marina Bay Stay",

"city": "Dubai",
"category": "Luxury",
"rating": 4.8,
"rooms_available": 12,
"price_per_night": 18000,
"amenities": ["wifi", "pool", "spa", "sea_view"],
"contact": {
"phone": "9876500012",
"email": "marinabay@mail.com"
}
},
{
"hotel_id": 203,
"hotel_name": "Budget Inn",
"city": "Delhi",
"category": "Budget",
"rating": 3.9,
"rooms_available": 40,
"price_per_night": 2200,
"amenities": ["wifi"],
"contact": {
"phone": null,
"email": "budgetinn@mail.com"
}
},
{
"hotel_id": 204,
"hotel_name": "Hill View Resort",
"city": "Kochi",
"category": "Resort",
"rating": 4.5,
"rooms_available": 18,
"price_per_night": 7500,
"amenities": ["wifi", "breakfast", "pool"],
"contact": {
"phone": "9876500014",
"email": null
}
},
{
"hotel_id": 205,
"hotel_name": "Skyline Suites",
"city": "London",
"category": "Luxury",

"rating": 4.7,
"rooms_available": 8,
"price_per_night": 22000,
"amenities": ["wifi", "breakfast", "spa"],
"contact": {
"phone": "9876500015",
"email": "skyline@mail.com"
}
}
]

Writing hotels.json


In [ ]:
hotels_df = spark.read.option(
"multiline",
"true"
).json("hotels.json")
hotels_df.show(truncate=False)
hotels_df.printSchema()

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [ ]:
hotels_df.show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [ ]:
hotels_df.select("hotel_name", "city", "rating").show(truncate=False)

+----------------+---------+------+
|hotel_name      |city     |rating|
+----------------+---------+------+
|Pearl Grand     |Hyderabad|4.4   |
|Marina Bay Stay |Dubai    |4.8   |
|Budget Inn      |Delhi    |3.9   |
|Hill View Resort|Kochi    |4.5   |
|Skyline Suites  |London   |4.7   |
+----------------+---------+------+



In [ ]:
hotels_df.filter(F.col("category") == "Luxury").show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [ ]:
hotels_df.filter(F.col("rating") > 4.5).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [ ]:
hotels_df.filter(F.col("rooms_available") > 15).show(truncate=False)

+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities              |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym] |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi]                 |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]|Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500           |4.5   |18             |
+-----------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------

In [ ]:
hotels_df.filter(F.col("price_per_night") > 10000).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [ ]:
hotels_df.filter(F.col("city").isin("Dubai", "London")).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [ ]:
hotels_df.filter(F.col("contact.phone").isNull()).show(truncate=False)

+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+
|amenities|category|city |contact                   |hotel_id|hotel_name|price_per_night|rating|rooms_available|
+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+
|[wifi]   |Budget  |Delhi|{budgetinn@mail.com, NULL}|203     |Budget Inn|2200           |3.9   |40             |
+---------+--------+-----+--------------------------+--------+----------+---------------+------+---------------+



In [ ]:
hotels_df.filter(F.col("contact.email").isNull()).show(truncate=False)

+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+
|amenities              |category|city |contact           |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, pool]|Resort  |Kochi|{NULL, 9876500014}|204     |Hill View Resort|7500           |4.5   |18             |
+-----------------------+--------+-----+------------------+--------+----------------+---------------+------+---------------+



In [ ]:
hotels_df.select("hotel_name", "contact.phone", "contact.email").show(truncate=False)

+----------------+----------+-------------------+
|hotel_name      |phone     |email              |
+----------------+----------+-------------------+
|Pearl Grand     |9876500011|pearlgrand@mail.com|
|Marina Bay Stay |9876500012|marinabay@mail.com |
|Budget Inn      |NULL      |budgetinn@mail.com |
|Hill View Resort|9876500014|NULL               |
|Skyline Suites  |9876500015|skyline@mail.com   |
+----------------+----------+-------------------+



In [ ]:
hotels_df.filter(F.array_contains(F.col("amenities"), "wifi")).show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [ ]:
hotels_df.filter(F.array_contains(F.col("amenities"), "spa")).show(truncate=False)

+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|amenities                  |category|city  |contact                         |hotel_id|hotel_name     |price_per_night|rating|rooms_available|
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai |{marinabay@mail.com, 9876500012}|202     |Marina Bay Stay|18000          |4.8   |12             |
|[wifi, breakfast, spa]     |Luxury  |London|{skyline@mail.com, 9876500015}  |205     |Skyline Suites |22000          |4.7   |8              |
+---------------------------+--------+------+--------------------------------+--------+---------------+---------------+------+---------------+



In [ ]:
hotels_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Kochi|    1|
|   London|    1|
|    Delhi|    1|
|Hyderabad|    1|
|    Dubai|    1|
+---------+-----+



In [ ]:
hotels_df.groupBy("category").count().show()

+--------+-----+
|category|count|
+--------+-----+
|  Resort|    1|
|  Budget|    1|
|Business|    1|
|  Luxury|    2|
+--------+-----+



In [ ]:
hotels_df.groupBy("category").agg(F.avg("rating").alias("avg_rating")).show()

+--------+----------+
|category|avg_rating|
+--------+----------+
|  Resort|       4.5|
|  Budget|       3.9|
|Business|       4.4|
|  Luxury|      4.75|
+--------+----------+



In [ ]:
hotels_df.groupBy("city").agg(F.avg("price_per_night").alias("avg_price")).show()

+---------+---------+
|     city|avg_price|
+---------+---------+
|    Kochi|   7500.0|
|   London|  22000.0|
|    Delhi|   2200.0|
|Hyderabad|   4500.0|
|    Dubai|  18000.0|
+---------+---------+



In [ ]:
max_price_df = hotels_df.agg(F.max("price_per_night").alias("max_price"))
max_price_df.show()

+---------+
|max_price|
+---------+
|    22000|
+---------+



In [ ]:
hotels_df = hotels_df.withColumn("total_potential_revenue", F.col("rooms_available") * F.col("price_per_night"))
hotels_df.show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|total_potential_revenue|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |112500                 |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |216000                 |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40       

In [ ]:
hotels_df.orderBy(F.col("rating").desc()).show(truncate=False)

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|total_potential_revenue|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+-----------------------+
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |216000                 |
|[wifi, breakfast, spa]     |Luxury  |London   |{skyline@mail.com, 9876500015}   |205     |Skyline Suites  |22000          |4.7   |8              |176000                 |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500           |4.5   |18       

In [ ]:
flat_df = hotels_df.select(
    F.col("hotel_id"),
    F.col("hotel_name"),
    F.col("city"),
    F.col("category"),
    F.col("rating"),
    F.col("rooms_available"),
    F.col("price_per_night"),
    F.col("total_potential_revenue"),
    F.concat_ws(", ", F.col("amenities")).alias("amenities_list"), # Array to single flat CSV string
    F.col("contact.phone").alias("contact_phone"),                  # Struct extract
    F.col("contact.email").alias("contact_email")                   # Struct extract
)

# Render sample presentation of flattened dataset structure
flat_df.show(truncate=False)

# Save directly into current Colab local environment partition block
flat_df.write.mode("overwrite").parquet("hotels_flattened.parquet")
print("Flattened Parquet generation finished successfully!")

+--------+----------------+---------+--------+------+---------------+---------------+-----------------------+-------------------------+-------------+-------------------+
|hotel_id|hotel_name      |city     |category|rating|rooms_available|price_per_night|total_potential_revenue|amenities_list           |contact_phone|contact_email      |
+--------+----------------+---------+--------+------+---------------+---------------+-----------------------+-------------------------+-------------+-------------------+
|201     |Pearl Grand     |Hyderabad|Business|4.4   |25             |4500           |112500                 |wifi, breakfast, gym     |9876500011   |pearlgrand@mail.com|
|202     |Marina Bay Stay |Dubai    |Luxury  |4.8   |12             |18000          |216000                 |wifi, pool, spa, sea_view|9876500012   |marinabay@mail.com |
|203     |Budget Inn      |Delhi    |Budget  |3.9   |40             |2200           |88000                  |wifi                     |NULL         |b